# Import

In [3]:
import os
import random
import glob
import re
from datetime import datetime

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from tqdm import tqdm

import optuna
from optuna.integration import PyTorchLightningPruningCallback
import warnings
warnings.filterwarnings('ignore')


# Fixed RandomSeed & Setting Hyperparameter

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

In [5]:
# 기본 하이퍼파라미터 (OPTUNA로 튜닝할 예정)
LOOKBACK = 28
PREDICT = 7
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_TRIALS = 50  # OPTUNA 시행 횟수

# Data Load and Preprocessing

In [6]:
'''
Load dataset and preprocessing
-> train | test | submission | prediction
'''

# train data
train_data = pd.read_csv('./dataset/train/train.csv')
# ['date'] -> datetime
train_data['date'] = pd.to_datetime(train_data['date'], format='%Y-%m-%d')
# ordinal date feature
train_data['date_ordinal'] = train_data['date'].map(datetime.toordinal)
# store_menu_id
train_data['store_menu_id'] = train_data['store'] + "_" + train_data['menu']

# test data
for i in range(0, 10):
    test = pd.read_csv(f"./dataset/test/TEST_0{i}.csv")
    test['date'] = pd.to_datetime(test['date'], format='%Y-%m-%d')
    test['date_ordinal'] = test['date'].map(datetime.toordinal)
    test['store_menu_id'] = test['store'] + "_" + test['menu']
    # test_data_{i} for all test datasets
    globals()[f'test_data_{i}'] = test

# submission format
submission = pd.read_csv("./result/sample_submission_date.csv")

# Prediction result
all_preds = []

# Define GRU Model

In [7]:
class MultiOutputGRU(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, output_dim=7, dropout=0.1):
        super(MultiOutputGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, 
                         batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out)
        return self.fc(out[:, -1, :])  # (B, output_dim)

# OPTUNA Objective Function

In [8]:
def create_sequences(data, lookback=28, predict=7):
    """시계열 데이터를 시퀀스로 변환"""
    X, y = [], []
    for i in range(len(data) - lookback - predict + 1):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback:i+lookback+predict, 0])
    return np.array(X), np.array(y)

def objective(trial):
    # 하이퍼파라미터 제안
    hidden_dim = trial.suggest_int('hidden_dim', 32, 256, step=32)
    num_layers = trial.suggest_int('num_layers', 1, 4)
    dropout = trial.suggest_float('dropout', 0.0, 0.5)
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32, 64])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    epochs = trial.suggest_int('epochs', 20, 100, step=10)
    
    # 검증을 위한 작은 샘플 선택 (전체 데이터로 하면 시간이 너무 오래 걸림)
    sample_stores = train_data['store_menu_id'].unique()[:20]  # 처음 20개 store_menu만 사용
    sample_data = train_data[train_data['store_menu_id'].isin(sample_stores)].copy()
    
    total_loss = 0
    valid_stores = 0
    
    for store_menu, group in sample_data.groupby('store_menu_id'):
        store_train = group.sort_values('date').copy()
        if len(store_train) < LOOKBACK + PREDICT + 10:  # 최소 데이터 요구량
            continue
            
        # 훈련/검증 분할
        train_size = int(len(store_train) * 0.8)
        train_part = store_train.iloc[:train_size]
        valid_part = store_train.iloc[train_size:]
        
        if len(valid_part) < PREDICT:
            continue
            
        # 정규화
        features = ['sales']
        scaler = MinMaxScaler()
        train_vals = scaler.fit_transform(train_part[features].values)
        valid_vals = scaler.transform(valid_part[features].values)
        
        # 시퀀스 생성
        X_train, y_train = create_sequences(train_vals, LOOKBACK, PREDICT)
        if len(X_train) == 0:
            continue
            
        X_train = torch.tensor(X_train).float().to(DEVICE)
        y_train = torch.tensor(y_train).float().to(DEVICE)
        
        # 모델 생성 및 훈련
        model = MultiOutputGRU(input_dim=1, hidden_dim=hidden_dim, 
                              num_layers=num_layers, output_dim=PREDICT, 
                              dropout=dropout).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.MSELoss()
        
        model.train()
        for epoch in range(epochs):
            idx = torch.randperm(len(X_train))
            epoch_loss = 0
            num_batches = 0
            
            for i in range(0, len(X_train), batch_size):
                batch_idx = idx[i:i+batch_size]
                X_batch, y_batch = X_train[batch_idx], y_train[batch_idx]
                
                optimizer.zero_grad()
                output = model(X_batch)
                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                num_batches += 1
                
            # Early stopping for trial pruning
            if epoch % 10 == 0:
                avg_loss = epoch_loss / max(num_batches, 1)
                trial.report(avg_loss, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
        
        # 검증
        model.eval()
        if len(valid_vals) >= LOOKBACK:
            recent_vals = valid_vals[-LOOKBACK:]
            x_input = torch.tensor([recent_vals]).float().to(DEVICE)
            
            with torch.no_grad():
                pred_scaled = model(x_input).squeeze().cpu().numpy()
            
            # 역변환
            pred_original = scaler.inverse_transform(
                pred_scaled.reshape(-1, 1)
            ).flatten()
            
            # 실제 값과 비교 (가능한 만큼)
            actual_vals = valid_part['sales'].values
            compare_len = min(len(pred_original), len(actual_vals))
            
            if compare_len > 0:
                mse = np.mean((pred_original[:compare_len] - actual_vals[:compare_len])**2)
                total_loss += mse
                valid_stores += 1
    
    if valid_stores == 0:
        return float('inf')
    
    return total_loss / valid_stores

# OPTUNA Hyperparameter Tuning

In [9]:
# OPTUNA 스터디 생성 및 실행
study = optuna.create_study(direction='minimize', 
                           pruner=optuna.pruners.MedianPruner())

print("하이퍼파라미터 튜닝 시작...")
study.optimize(objective, n_trials=N_TRIALS)

print("\n=== 최적 하이퍼파라미터 ===")
print(f"Best value: {study.best_value}")
print(f"Best params: {study.best_params}")

# 최적 파라미터 저장
best_params = study.best_params

[I 2025-08-05 19:20:40,990] A new study created in memory with name: no-name-2508942f-bbc5-4142-b5a6-5df89b770dda


하이퍼파라미터 튜닝 시작...


[W 2025-08-05 19:22:26,775] Trial 0 failed with parameters: {'hidden_dim': 32, 'num_layers': 3, 'dropout': 0.3711886973427224, 'batch_size': 16, 'learning_rate': 0.003731802514794691, 'epochs': 80} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/4f/fh9gn06n6hd15fkrmf2t7w3h0000gn/T/ipykernel_79838/2863404369.py", line 72, in objective
    loss.backward()
  File "/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/torch/_tensor.py", line 521, in backward
    torch.autograd.backward(
  File "/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/torch/autograd/__init__.py", line 289, in backward
    _engine_run_backward(
  File "/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/torch/autograd/graph.py", line 768, in _engine_run_backward
    

KeyboardInterrupt: 

# Train with Best Parameters

In [ ]:
def train_gru_with_best_params(train_df, best_params):
    """최적 파라미터로 전체 데이터에 대해 GRU 모델 훈련"""
    trained_models = {}
    
    print(f"최적 파라미터로 전체 모델 훈련 시작: {best_params}")
    
    for store_menu, group in tqdm(train_df.groupby(['store_menu_id']), desc='Training GRU'):
        store_train = group.sort_values('date').copy()
        if len(store_train) < LOOKBACK + PREDICT:
            continue

        features = ['sales']
        scaler = MinMaxScaler()
        store_train[features] = scaler.fit_transform(store_train[features])
        train_vals = store_train[features].values  # shape: (N, 1)

        # 시퀀스 구성
        X_train, y_train = [], []
        for i in range(len(train_vals) - LOOKBACK - PREDICT + 1):
            X_train.append(train_vals[i:i+LOOKBACK])
            y_train.append(train_vals[i+LOOKBACK:i+LOOKBACK+PREDICT, 0])

        if len(X_train) == 0:
            continue
            
        X_train = torch.tensor(X_train).float().to(DEVICE)
        y_train = torch.tensor(y_train).float().to(DEVICE)

        # 최적 파라미터로 모델 생성
        model = MultiOutputGRU(
            input_dim=1, 
            hidden_dim=best_params['hidden_dim'],
            num_layers=best_params['num_layers'], 
            output_dim=PREDICT,
            dropout=best_params['dropout']
        ).to(DEVICE)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=best_params['learning_rate'])
        criterion = nn.MSELoss()

        model.train()
        for epoch in range(best_params['epochs']):
            idx = torch.randperm(len(X_train))
            for i in range(0, len(X_train), best_params['batch_size']):
                batch_idx = idx[i:i+best_params['batch_size']]
                X_batch, y_batch = X_train[batch_idx], y_train[batch_idx]
                
                optimizer.zero_grad()
                output = model(X_batch)
                loss = criterion(output, y_batch)
                loss.backward()
                optimizer.step()

        trained_models[store_menu] = {
            'model': model.eval(),
            'scaler': scaler,
            'last_sequence': train_vals[-LOOKBACK:]  # (28, 1)
        }

    return trained_models

In [ ]:
# 최적 파라미터로 전체 모델 훈련
trained_models = train_gru_with_best_params(train_data, best_params)

# Prediction

In [ ]:
def predict_gru(test_df, trained_models, test_prefix: str):
    """GRU 모델로 예측"""
    results = []

    for store_menu, store_test in test_df.groupby(['store_menu_id']):
        key = store_menu
        if key not in trained_models:
            # 훈련된 모델이 없으면 0으로 예측
            pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]
            for d in pred_dates:
                results.append({
                    'date': d,
                    'store_menu_id': store_menu,
                    'sales': 0
                })
            continue

        model = trained_models[key]['model']
        scaler = trained_models[key]['scaler']

        store_test_sorted = store_test.sort_values('date')
        recent_vals = store_test_sorted['sales'].values[-LOOKBACK:]
        if len(recent_vals) < LOOKBACK:
            # 데이터가 부족하면 0으로 예측
            pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]
            for d in pred_dates:
                results.append({
                    'date': d,
                    'store_menu_id': store_menu,
                    'sales': 0
                })
            continue

        # 정규화
        recent_vals_scaled = scaler.transform(recent_vals.reshape(-1, 1))
        x_input = torch.tensor([recent_vals_scaled]).float().to(DEVICE)

        with torch.no_grad():
            pred_scaled = model(x_input).squeeze().cpu().numpy()

        # 역변환
        restored = []
        for i in range(PREDICT):
            dummy = np.zeros((1, 1))
            dummy[0, 0] = pred_scaled[i]
            restored_val = scaler.inverse_transform(dummy)[0, 0]
            restored.append(max(restored_val, 0))  # 음수 방지

        # 예측일자: TEST_00+1일 ~ TEST_00+7일
        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]

        for d, val in zip(pred_dates, restored):
            results.append({
                'date': d,
                'store_menu_id': store_menu,
                'sales': val
            })

    return pd.DataFrame(results)

In [ ]:
# 모든 테스트 데이터에 대해 예측 수행
all_preds = []

for i in range(10):
    test_df = globals()[f'test_data_{i}']
    test_prefix = f'TEST_0{i}'
    
    print(f"Predicting for {test_prefix}...")
    pred_df = predict_gru(test_df, trained_models, test_prefix)
    all_preds.append(pred_df)
    
full_pred_df = pd.concat(all_preds, ignore_index=True)
print(f"\n전체 예측 완료: {len(full_pred_df)} rows")

# Submission

In [ ]:
def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    """예측 결과를 제출 형식으로 변환"""
    # (date, store_menu_id) → sales 딕셔너리로 변환
    pred_dict = dict(zip(
        zip(pred_df['date'], pred_df['store_menu_id']),
        pred_df['sales']
    ))

    final_df = sample_submission.copy()

    for row_idx in final_df.index:
        date = final_df.loc[row_idx, 'date']  # 제출 형식의 date 컬럼
        for col in final_df.columns[1:]:  # store_menu_id들
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)

    return final_df

In [ ]:
# 제출 파일 생성
submission_result = convert_to_submission_format(full_pred_df, submission)
submission_result.to_csv('./result/gru_optuna_submission.csv', index=False, encoding='utf-8-sig')

print("제출 파일 생성 완료: gru_optuna_submission.csv")
print(f"제출 파일 크기: {submission_result.shape}")
print(f"\n최적 하이퍼파라미터: {best_params}")

# Summary

In [ ]:
# 결과 요약
print("\n=== GRU + OPTUNA 예측 결과 요약 ===")
print(f"훈련된 모델 수: {len(trained_models)}")
print(f"총 예측 샘플 수: {len(full_pred_df)}")
print(f"평균 예측값: {full_pred_df['sales'].mean():.2f}")
print(f"예측값 표준편차: {full_pred_df['sales'].std():.2f}")
print(f"\n최적 하이퍼파라미터:")
for key, value in best_params.items():
    print(f"  {key}: {value}")